# VAE inpainting training


In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Clone repo and install

In [ ]:
%cd /content
!git clone https://github.com/Mcfish99/Intro-to-Deep-Learning-Final-Project.git
%cd Intro-to-Deep-Learning-Final-Project
!git pull

In [ ]:
!pip install -q -r requirements.txt
!pip install -q "numpy<2"

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB VRAM")

### Dataset

Copy from Drive to the local SSD. Training off Drive directly is painfully slow.

In [ ]:
import os

DRIVE_DATA = "/content/drive/MyDrive/celeba_hq/train"  # change this to your actual path
DATA_DIR = "/content/data/train"

if not os.path.exists(DATA_DIR):
    !mkdir -p /content/data
    !cp -r {DRIVE_DATA} /content/data/

n = len([f for f in os.listdir(DATA_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
print(f"{n} images")

If you don't have CelebA-HQ on Drive yet, grab the 256x256 version from Kaggle (upload kaggle.json first):

```
!pip install -q kaggle
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d badasstechie/celebahq-resized-256x256 -p /content/data --unzip
```

In [ ]:
import os, shutil, glob

TEST_SUBSET = "/content/test_subset"
os.makedirs(TEST_SUBSET, exist_ok=True)

for src in sorted(glob.glob(f"{DATA_DIR}/*.jpg") + glob.glob(f"{DATA_DIR}/*.png"))[:16]:
    dst = os.path.join(TEST_SUBSET, os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copy(src, dst)

print(f"{len(os.listdir(TEST_SUBSET))} images")

In [ ]:
!python -m src.VAE.train \
    --data-dir /content/test_subset \
    --output-dir /tmp/vae_overfit \
    --overfit --overfit-steps 200 \
    --batch-size 8 \
    --log-every 20

### Training

In [ ]:
# Save to Drive so checkpoints survive disconnects
CKPT_DIR = "/content/drive/MyDrive/inpainting_checkpoints/vae"
!mkdir -p {CKPT_DIR}

In [ ]:
!python -m src.VAE.train \
    --data-dir {DATA_DIR} \
    --output-dir {CKPT_DIR} \
    --epochs 30 \
    --batch-size 32 \
    --num-workers 8 \
    --image-size 256 \
    --lr 2e-4 \
    --beta-kl 0.01 \
    --lambda-perc 0.1 \
    --log-every 100

To keep Colab from disconnecting on idle, open browser devtools (F12) and paste into the console:

```js
setInterval(() => document.querySelector("colab-connect-button")?.shadowRoot?.querySelector("#connect")?.click(), 60000);
```

### Peek at samples during training

In [ ]:
from IPython.display import Image, display
import glob

samples = sorted(glob.glob(f"{CKPT_DIR}/samples/*.png"))
print(f"{len(samples)} epochs saved")
if samples:
    display(Image(samples[-1]))

### Visualize results across mask sizes

In [ ]:
import sys
sys.path.insert(0, "/content/Intro-to-Deep-Learning-Final-Project")

import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from src.VAE.inpainter import VAEInpainter
from src.shared.dataset import CelebAHQDataset
from src.shared.mask import batch_masks

vae = VAEInpainter(ckpt_path=f"{CKPT_DIR}/vae_latest.pt", device="cuda")

ds = CelebAHQDataset(DATA_DIR, image_size=256)
x = next(iter(DataLoader(ds, batch_size=4, shuffle=True))).cuda()

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for r, regime in enumerate(["small", "medium", "large"]):
    mask = batch_masks(x.size(0), 256, 256, condition=regime).cuda()
    y = x * (1 - mask)
    xh = vae.inpaint(y, mask)
    for c, (img, t) in enumerate([(x[0], f"GT ({regime})"), (y[0], "Corrupted"), (xh[0], "VAE")]):
        axes[r, c].imshow(((img.detach().cpu() + 1) / 2).clamp(0, 1).permute(1, 2, 0))
        axes[r, c].set_title(t); axes[r, c].axis("off")

plt.tight_layout(); plt.show()

### Diverse completions

Same corrupted input, different plausible outputs — VAE can do this, deterministic methods can't. Good figure for the report.

In [ ]:
x_one = x[:1]
mask_one = batch_masks(1, 256, 256, condition="medium").cuda()
y_one = x_one * (1 - mask_one)

samples = vae.sample_diverse(y_one, mask_one, n_samples=6)

fig, axes = plt.subplots(1, 8, figsize=(20, 3))
axes[0].imshow(((x_one[0].detach().cpu() + 1) / 2).clamp(0, 1).permute(1, 2, 0))
axes[0].set_title("GT"); axes[0].axis("off")
axes[1].imshow(((y_one[0].detach().cpu() + 1) / 2).clamp(0, 1).permute(1, 2, 0))
axes[1].set_title("Corrupted"); axes[1].axis("off")
for i, s in enumerate(samples):
    axes[i+2].imshow(((s[0].detach().cpu() + 1) / 2).clamp(0, 1).permute(1, 2, 0))
    axes[i+2].set_title(f"z~{i+1}"); axes[i+2].axis("off")

plt.tight_layout(); plt.show()